# 🩺 Domácí úkol (Session 2): XGBoost – Klasifikace diabetu
**Dataset:** `diabetes.csv` (768 pacientek kmene Pima, 9 medicínských atributů)  
**Cíl úlohy:** Konstrukce a optimalizace klasifikačního modelu gradientního boostingu (`XGBClassifier` z knihovny `xgboost`) pro predikci diagnózy diabetu (`Outcome`), včetně detailní inspekce přes `ydata_profiling`, rozboru biologicky nemožných nulových hodnot, rozdělení 80:20 (`random_state=21`), hyperparametrického ladění přes `GridSearchCV` s optimalizací na metriku **`precision`** a vyhodnocení přes `classification_report`.

---
### 📋 Kroky zadání:
1. **Načtení dat:** Stáhněte soubor `diabetes.csv` z materiálů a načtěte jej pomocí knihovny Pandas.
2. **Import tříd:** Importujte potřebné třídy pro XGBoost model z knihovny `xgboost` a metriky ze `scikit-learn`.
3. **Data Profiling report:** Zobrazte souhrnný report pomocí balíčku `ydata_profiling`:
   ```python
   !pip install ydata_profiling
   import ydata_profiling as ydp
   diabetes_report = ydp.ProfileReport(diabetes_df)
   diabetes_report
   ```
4. **Analýza reportu:** Vyhodnoťte rozdělení proměnných, korelaci a nulové hodnoty (zejména u glukózy, tlaku, inzulinu a BMI).
5. **Příprava dat:** Ošetření dat před modelováním (chybějící hodnoty, škálování / normalizace).
6. **Train-Test Split:** Rozdělte dataset v poměru 80:20 se stanoveným `random_state=21`.
7. **Instance modelu:** Vytvořte instanci modelu `xgb_classifier` se specifikovaným parametrem `objective`.
8. **Mřížka parametrů:** Definujte slovník `params` pro klíče `max_depth`, `n_estimators`, `gamma`, `objective` a `learning_rate` (2 až 4 hodnoty na klíč).
9. **GridSearchCV:** Vytvořte objekt `GridSearchCV` pro výběr nejlepší kombinace s optimalizovanou metrikou `scoring='precision'`.
10. **Trénování:** Natrénujte nejlepší model na trénovacích datech `X_train, y_train`.
11. **Predikce:** Proveďte odhady na testovací množině `X_test`.
12. **Metriky:** Vygenerujte souhrnný report metrik `classification_report` pro testovací sadu.


In [ ]:
# Krok 1 & 2: Import knihoven a načtení dat
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import xgboost as xgb
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import classification_report, confusion_matrix, precision_score, recall_score, f1_score, accuracy_score, roc_auc_score

# Načtení datasetu diabetes
try:
    diabetes_df = pd.read_csv('diabetes.csv')
except FileNotFoundError:
    try:
        diabetes_df = pd.read_csv('data/diabetes.csv')
    except FileNotFoundError:
        diabetes_df = pd.read_csv('data/MAL_downloadable materials_session 2/Homework/diabetes.csv')

print(f"Dataset úspěšně načten! Tvar: {diabetes_df.shape}")
diabetes_df.head()


### 📊 Krok 3 & 4: Souhrnný report dat (ydata_profiling) & Analýza nulových hodnot
Zadání explicitně vyžaduje generování profilovacího reportu pomocí balíčku `ydata_profiling`.  
V lékařských datech je klíčové věnovat pozornost **nulovým hodnotám**:
- Hodnoty `Glucose == 0`, `BloodPressure == 0`, `SkinThickness == 0`, `Insulin == 0` a `BMI == 0` jsou u živého člověka fyziologicky nemožné! Jedná se o chybějící měření (Missing Values) zakódovaná jako nuly.


In [ ]:
# Krok 3: Generování profilovacího reportu (dle zadání)
# !pip install ydata_profiling

try:
    import ydata_profiling as ydp
    # diabetes_report = ydp.ProfileReport(diabetes_df, title="Diabetes Profiling Report", minimal=True)
    # diabetes_report.to_file("diabetes_report.html")
    print("Knihovna ydata_profiling je dostupná. Report lze vygenerovat přes diabetes_report.")
except ImportError:
    print("Tip pro Colab / Jupyter: Nainstalujte knihovnu příkazem '!pip install ydata_profiling'")

# Inspekce biologických nul
zero_cols = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
print("\nZastoupení fyziologicky nemožných nul v jednotlivých atributech:")
for col in zero_cols:
    cnt = (diabetes_df[col] == 0).sum()
    pct = (diabetes_df[col] == 0).mean() * 100
    print(f" - {col:15s}: {cnt:3d} nul ({pct:5.1f} %)")


### ✂️ Krok 5 & 6: Příprava dat a rozdělení na Train/Test (80:20, random_state=21)
Oddělíme vysvětlující proměnné `X` a cílovou proměnnou `y` (`Outcome`).  
Dle zadání dělíme data v poměru **80:20** se zadaným **`random_state=21`**.


In [ ]:
# Krok 5 & 6: Train-Test Split (80:20, random_state=21)
X = diabetes_df.drop(columns=['Outcome'])
y = diabetes_df['Outcome']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=21
)

print(f"Trénovací sada: {X_train.shape[0]} pacientek, {X_train.shape[1]} atributů")
print(f"Testovací sada : {X_test.shape[0]} pacientek, {X_test.shape[1]} atributů")


### ⚡ Kroky 7 & 8: Instance modelu XGBoost a mřížka hyperparametrů
Inicializujeme `xgb.XGBClassifier` se zadaným parametrem `objective='binary:logistic'` a `eval_metric='logloss'`.  
Definujeme mřížku parametrů `params` pro klíče:
- `max_depth`: [3, 4, 5]
- `n_estimators`: [50, 100, 150]
- `gamma`: [0.0, 0.1, 0.5]
- `learning_rate`: [0.05, 0.1, 0.2]
- `objective`: ['binary:logistic']


In [ ]:
# Krok 7 & 8: Instance modelu a mřížka hyperparametrů
xgb_classifier = xgb.XGBClassifier(
    objective='binary:logistic',
    random_state=21,
    eval_metric='logloss',
    n_jobs=-1
)

params = {
    'max_depth': [3, 4, 5],
    'n_estimators': [50, 100, 150],
    'gamma': [0.0, 0.1, 0.5],
    'learning_rate': [0.05, 0.1, 0.2],
    'objective': ['binary:logistic']
}


### 🔍 Kroky 9 & 10: GridSearchCV s optimalizací na metriku `precision` a dotrénování
Vytvoříme objekt `GridSearchCV` s parametrem `scoring='precision'` a provedeme 5-násobnou křížovou validaci.  
Optimalizace na `precision` cílí na to, aby v případě, kdy model označí pacientku za diabetičku, byla diagnóza co nejspolehlivější (minimalizace falešně pozitivních nálezů).


In [ ]:
# Kroky 9 & 10: Spuštění GridSearchCV
grid_search = GridSearchCV(
    estimator=xgb_classifier,
    param_grid=params,
    scoring='precision',
    cv=5,
    n_jobs=-1,
    return_train_score=True
)

print("Spouštím GridSearchCV (optimalizace na metriku precision)...")
grid_search.fit(X_train, y_train)

print(f"\nNejlepší parametry: {grid_search.best_params_}")
print(f"Nejlepší CV Precision (5-fold): {grid_search.best_score_:.4f}")

# Nejlepší model
best_xgb = grid_search.best_estimator_


### 📋 Kroky 11 & 12: Predikce na testovací sadě & Souhrnný Classification Report
Vygenerujeme predikce pro testovací data `X_test` a zobrazíme detailní klasifikační report.


In [ ]:
# Kroky 11 & 12: Predikce a vyhodnocení
y_pred = best_xgb.predict(X_test)
y_prob = best_xgb.predict_proba(X_test)[:, 1]

print("=" * 60)
print("📊 TEST SET CLASSIFICATION REPORT (XGBOOST)")
print("=" * 60)
print(classification_report(y_test, y_pred, target_names=['Zdravý (0)', 'Diabetik (1)']))
print("=" * 60)

# Doplňující metriky
acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)

print(f"Celková přesnost (Accuracy) : {acc*100:.2f} %")
print(f"Preciznost třídy 1 (Precision): {prec*100:.2f} %")
print(f"Záchyt třídy 1 (Recall)      : {rec*100:.2f} %")
print(f"F1-skóre třídy 1             : {f1:.4f}")
print(f"Plocha pod ROC křivkou (AUC) : {auc:.4f}")


### 💡 Analýza důležitosti příznaků (XGBoost Gain vs. Weight)
Které medicínské ukazatele mají největší vliv na rozhodování posilovaných rozhodovacích stromů?


In [ ]:
# Důležitost příznaků
booster = best_xgb.get_booster()
gain_dict = booster.get_score(importance_type='gain')
imp_series = pd.Series(gain_dict).sort_values(ascending=False)

plt.figure(figsize=(10, 5))
sns.barplot(x=imp_series.values, y=imp_series.index, palette='magma')
plt.title('Důležitost příznaků dle XGBoost Gain (přínos ke snížení logloss)')
plt.xlabel('Gain')
plt.ylabel('Atribut')
plt.tight_layout()
plt.show()
